# FM Action-Stepper + PPO Maze Solver

This version **replaces the sketcher** with a Flow-Matching action-stepper.
The FM model learns *action-conditioned step transitions* (up/down/left/right) that extend the red path.
PPO then uses these learned action steps iteratively to reach the goal.

The red path **is the agent**; no separate yellow dot.


In [1]:
%cd /Users/masha/Documents/visual-reasoning


/Users/masha/Documents/visual-reasoning


In [2]:
import math
import random
from collections import deque
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import gymnasium as gym

from stable_baselines3 import PPO


Config + device helpers.

In [3]:
def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def set_seed(seed: int = 0) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


DEVICE = get_device()
print("Device:", DEVICE)

RUN_FAST = False
IMG_SIZE = 64
MAZE_CELLS = 9

TRAIN_SAMPLES = 2000 if not RUN_FAST else 400
TEST_SAMPLES = 400 if not RUN_FAST else 80
BATCH_SIZE = 32
EPOCHS = 40 if not RUN_FAST else 12
LR = 2e-4

PPO_STEPS = 100_000 if not RUN_FAST else 20_000
EVAL_EPISODES = 30 if not RUN_FAST else 8
MAX_EP_STEPS = 180


Device: mps


Maze generation + utilities.

In [4]:
def generate_maze(cells_w: int, cells_h: int, rng: random.Random) -> np.ndarray:
    """Generate a perfect maze with DFS backtracking. Returns grid with 1=wall, 0=free."""
    grid = np.ones((cells_h * 2 + 1, cells_w * 2 + 1), dtype=np.uint8)
    visited = np.zeros((cells_h, cells_w), dtype=bool)

    stack = [(0, 0)]
    visited[0, 0] = True
    grid[1, 1] = 0

    while stack:
        x, y = stack[-1]
        neighbors = []
        for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nx, ny = x + dx, y + dy
            if 0 <= nx < cells_w and 0 <= ny < cells_h and not visited[ny, nx]:
                neighbors.append((nx, ny, dx, dy))
        if neighbors:
            nx, ny, dx, dy = rng.choice(neighbors)
            grid[y * 2 + 1 + dy, x * 2 + 1 + dx] = 0
            grid[ny * 2 + 1, nx * 2 + 1] = 0
            visited[ny, nx] = True
            stack.append((nx, ny))
        else:
            stack.pop()

    return grid


def bfs_shortest_path(grid: np.ndarray, start: Tuple[int, int], goal: Tuple[int, int]) -> List[Tuple[int, int]]:
    h, w = grid.shape
    q = deque([start])
    prev = {start: None}

    while q:
        y, x = q.popleft()
        if (y, x) == goal:
            break
        for dy, dx in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            ny, nx = y + dy, x + dx
            if 0 <= ny < h and 0 <= nx < w and grid[ny, nx] == 0:
                if (ny, nx) not in prev:
                    prev[(ny, nx)] = (y, x)
                    q.append((ny, nx))

    if goal not in prev:
        return []

    path = []
    cur = goal
    while cur is not None:
        path.append(cur)
        cur = prev[cur]
    path.reverse()
    return path


def one_hot_point(shape: Tuple[int, int], pos: Tuple[int, int]) -> np.ndarray:
    m = np.zeros(shape, dtype=np.float32)
    m[pos[0], pos[1]] = 1.0
    return m


def path_to_trace(shape: Tuple[int, int], path: List[Tuple[int, int]], k: int) -> np.ndarray:
    trace = np.zeros(shape, dtype=np.float32)
    k = max(0, min(k, len(path) - 1))
    for y, x in path[: k + 1]:
        trace[y, x] = 1.0
    return trace


def action_from_step(a: Tuple[int, int], b: Tuple[int, int]) -> int:
    dy, dx = b[0] - a[0], b[1] - a[1]
    if dy == -1 and dx == 0:
        return 0  # up
    if dy == 1 and dx == 0:
        return 1  # down
    if dy == 0 and dx == -1:
        return 2  # left
    if dy == 0 and dx == 1:
        return 3  # right
    raise ValueError("Invalid step")


def resize_nn(t: torch.Tensor, size: int) -> torch.Tensor:
    if t.dim() == 2:
        t = t.unsqueeze(0).unsqueeze(0)
    elif t.dim() == 3:
        t = t.unsqueeze(0)
    out = F.interpolate(t, size=(size, size), mode="nearest")
    return out.squeeze(0)


def build_static_cond(grid: np.ndarray, start: Tuple[int, int], goal: Tuple[int, int], img_size: int) -> torch.Tensor:
    walls = grid.astype(np.float32)
    start_ch = one_hot_point(grid.shape, start)
    goal_ch = one_hot_point(grid.shape, goal)
    cond = np.stack([walls, start_ch, goal_ch], axis=0)
    return resize_nn(torch.tensor(cond).float(), img_size)


def build_action_cond(action: int, shape: Tuple[int, int], img_size: int) -> torch.Tensor:
    a = np.zeros((4, shape[0], shape[1]), dtype=np.float32)
    a[action, :, :] = 1.0
    return resize_nn(torch.tensor(a).float(), img_size)


Dataset: action-conditioned one-step transitions.

In [5]:
class MazeStepDataset(Dataset):
    def __init__(self, n_samples: int, maze_cells: int, img_size: int, seed: int = 0):
        self.n = n_samples
        self.maze_cells = maze_cells
        self.img_size = img_size
        self.rng = random.Random(seed)

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        grid = generate_maze(self.maze_cells, self.maze_cells, self.rng)
        start = (1, 1)
        goal = (grid.shape[0] - 2, grid.shape[1] - 2)
        path = bfs_shortest_path(grid, start, goal)
        if len(path) < 2:
            return self.__getitem__(idx + 1)

        k = self.rng.randint(0, len(path) - 2)
        action = action_from_step(path[k], path[k + 1])

        trace_t = path_to_trace(grid.shape, path, k)
        trace_next = path_to_trace(grid.shape, path, k + 1)
        delta = trace_next - trace_t

        cond_static = build_static_cond(grid, start, goal, self.img_size)
        cond_action = build_action_cond(action, grid.shape, self.img_size)
        cond = torch.cat([cond_static, cond_action], dim=0)

        trace_t = resize_nn(torch.tensor(trace_t).float(), self.img_size)
        delta = resize_nn(torch.tensor(delta).float(), self.img_size)

        return cond, trace_t, delta


train_loader = DataLoader(MazeStepDataset(TRAIN_SAMPLES, MAZE_CELLS, IMG_SIZE, seed=0),
                          batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(MazeStepDataset(TEST_SAMPLES, MAZE_CELLS, IMG_SIZE, seed=123),
                         batch_size=BATCH_SIZE, shuffle=False)


FM action-stepper model (predicts delta for each action).

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class MazeStepperFM(nn.Module):
    def __init__(self, cond_ch: int = 7, flow_dim: int = 32):
        super().__init__()
        self.inc = DoubleConv(1 + cond_ch, flow_dim)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(flow_dim, flow_dim * 2))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(flow_dim * 2, flow_dim * 4))

        self.up1 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.conv1 = DoubleConv(flow_dim * 6, flow_dim * 2)
        self.up2 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.conv2 = DoubleConv(flow_dim * 3, flow_dim)

        self.outc = nn.Conv2d(flow_dim, 1, kernel_size=1)

    def forward(self, trace_t: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        x = torch.cat([trace_t, cond], dim=1)
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x = self.conv1(torch.cat([self.up1(x3), x2], dim=1))
        x = self.conv2(torch.cat([self.up2(x), x1], dim=1))
        return self.outc(x)


stepper = MazeStepperFM(cond_ch=7, flow_dim=32).to(DEVICE)
optimizer = torch.optim.AdamW(stepper.parameters(), lr=LR)


Train the FM action-stepper.

In [ ]:
set_seed(0)
train_losses = []

for epoch in range(EPOCHS):
    stepper.train()
    epoch_loss = 0.0
    for cond, trace_t, delta in train_loader:
        cond = cond.to(DEVICE)
        trace_t = trace_t.to(DEVICE)
        delta = delta.to(DEVICE)

        pred = stepper(trace_t, cond)
        loss = F.mse_loss(pred, delta)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg = epoch_loss / max(1, len(train_loader))
    train_losses.append(avg)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1:02d} | loss {avg:.5f}")

plt.plot(train_losses)
plt.title("Action-stepper train loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.show()


Visualize the FM stepper building the red path (no yellow agent).

In [ ]:
@torch.no_grad()
def rollout_stepper(model: nn.Module, grid: np.ndarray, start: Tuple[int, int], goal: Tuple[int, int], path: List[Tuple[int, int]]):
    model.eval()
    cond_static = build_static_cond(grid, start, goal, IMG_SIZE).unsqueeze(0).to(DEVICE)
    trace = torch.zeros((1, 1, IMG_SIZE, IMG_SIZE), device=DEVICE)
    frames = []

    for i in range(len(path) - 1):
        action = action_from_step(path[i], path[i + 1])
        cond_action = build_action_cond(action, grid.shape, IMG_SIZE).unsqueeze(0).to(DEVICE)
        cond = torch.cat([cond_static, cond_action], dim=1)
        delta = model(trace, cond)
        trace = (trace + delta).clamp(0.0, 1.0)
        frames.append(trace.detach().cpu())

    return frames


# sample maze + path
rng = random.Random(42)
grid = generate_maze(MAZE_CELLS, MAZE_CELLS, rng)
start = (1, 1)
goal = (grid.shape[0] - 2, grid.shape[1] - 2)
path = bfs_shortest_path(grid, start, goal)

frames = rollout_stepper(stepper, grid, start, goal, path)

# render frames
def compose_rgb(cond: np.ndarray, trace: Optional[np.ndarray] = None) -> np.ndarray:
    walls = cond[0]
    start_ch = cond[1]
    goal_ch = cond[2]
    h, w = walls.shape
    img = np.ones((h, w, 3), dtype=np.float32)
    img[walls > 0.5] = 0.0
    img[start_ch > 0.5] = np.array([0.2, 1.0, 0.2])
    img[goal_ch > 0.5] = np.array([0.2, 0.2, 1.0])
    if trace is not None:
        img[trace > 0.5] = np.array([1.0, 0.2, 0.2])
    return img

cond_img = build_static_cond(grid, start, goal, IMG_SIZE)
idxs = np.linspace(0, len(frames) - 1, num=min(6, len(frames)), dtype=int)
fig, axes = plt.subplots(1, len(idxs), figsize=(2.6 * len(idxs), 2.6))
if len(idxs) == 1:
    axes = [axes]
for ax, i in zip(axes, idxs):
    trace = frames[i][0, 0].numpy()
    img = compose_rgb(cond_img.numpy(), trace=trace)
    ax.imshow(img)
    ax.set_title(f"Step {i}")
    ax.axis("off")
plt.tight_layout()
plt.show()


Gymnasium env: PPO chooses actions, FM stepper renders the red path. Agent position is tracked internally.

In [ ]:
class MazeEnvFMAction(gym.Env):
    metadata = {"render_modes": []}

    def __init__(
        self,
        stepper: nn.Module,
        maze_cells: int = 9,
        img_size: int = 64,
        max_steps: int = 180,
        device: Optional[torch.device] = None,
        seed: int = 0,
    ):
        super().__init__()
        self.stepper = stepper
        self.maze_cells = maze_cells
        self.img_size = img_size
        self.max_steps = max_steps
        self.device = device or get_device()
        self.rng = random.Random(seed)

        # channels: walls, start, goal, trace
        self.observation_space = gym.spaces.Box(
            low=0.0, high=1.0, shape=(4, img_size, img_size), dtype=np.float32
        )
        self.action_space = gym.spaces.Discrete(4)  # up, down, left, right

        self.grid = None
        self.start = None
        self.goal = None
        self.pos = None
        self.step_count = 0
        self.trace = None  # torch (1,1,H,W)
        self.cond_static = None  # torch (1,3,H,W)
        self.dist_map = None
        self.prev_dist = None

    def _compute_dist_map(self) -> np.ndarray:
        h, w = self.grid.shape
        dist = -np.ones((h, w), dtype=np.int32)
        gy, gx = self.goal
        if self.grid[gy, gx] == 1:
            return dist
        q = deque([(gy, gx)])
        dist[gy, gx] = 0
        while q:
            y, x = q.popleft()
            for dy, dx in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                ny, nx = y + dy, x + dx
                if 0 <= ny < h and 0 <= nx < w and self.grid[ny, nx] == 0 and dist[ny, nx] == -1:
                    dist[ny, nx] = dist[y, x] + 1
                    q.append((ny, nx))
        return dist

    def _obs(self) -> np.ndarray:
        trace_np = self.trace[0].detach().cpu().numpy()  # (1, H, W)
        cond_np = self.cond_static[0].detach().cpu().numpy()  # (3, H, W)
        obs = np.concatenate([cond_np, trace_np], axis=0)
        return obs.astype(np.float32)

    def reset(self, *, seed: Optional[int] = None, options: Optional[dict] = None):
        super().reset(seed=seed)
        if seed is not None:
            self.rng = random.Random(seed)

        grid = generate_maze(self.maze_cells, self.maze_cells, self.rng)
        start = (1, 1)
        goal = (grid.shape[0] - 2, grid.shape[1] - 2)

        self.grid = grid
        self.start = start
        self.goal = goal
        self.pos = start
        self.step_count = 0

        self.dist_map = self._compute_dist_map()
        self.prev_dist = int(self.dist_map[start[0], start[1]]) if self.dist_map is not None else 0

        cond_static = build_static_cond(grid, start, goal, self.img_size)
        self.cond_static = cond_static.unsqueeze(0).to(self.device)
        self.trace = torch.zeros((1, 1, self.img_size, self.img_size), device=self.device)

        return self._obs(), {}

    def step(self, action: int):
        y, x = self.pos
        dy, dx = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}[int(action)]
        ny, nx = y + dy, x + dx

        reward = -0.01
        if 0 <= ny < self.grid.shape[0] and 0 <= nx < self.grid.shape[1] and self.grid[ny, nx] == 0:
            self.pos = (ny, nx)
        else:
            reward -= 0.05

        # action-conditioned FM step for trace
        cond_action = build_action_cond(int(action), self.grid.shape, self.img_size).unsqueeze(0).to(self.device)
        cond = torch.cat([self.cond_static, cond_action], dim=1)
        with torch.no_grad():
            delta = self.stepper(self.trace, cond)
            self.trace = (self.trace + delta).clamp(0.0, 1.0)

        # ensure current position is on the red trace
        pos_ch = one_hot_point(self.grid.shape, self.pos)
        pos_ch = resize_nn(torch.tensor(pos_ch).float(), self.img_size).to(self.device)
        self.trace = torch.maximum(self.trace, pos_ch.unsqueeze(0))

        # distance shaping to goal (true position)
        dist = int(self.dist_map[self.pos[0], self.pos[1]]) if self.dist_map is not None else 0
        if dist >= 0:
            reward += (self.prev_dist - dist) * 0.05
            self.prev_dist = dist

        terminated = (self.pos == self.goal)
        if terminated:
            reward += 10.0

        self.step_count += 1
        truncated = self.step_count >= self.max_steps

        return self._obs(), float(reward), terminated, truncated, {}


Train PPO on FM action-stepper env.

In [ ]:
set_seed(0)

env = MazeEnvFMAction(
    stepper=stepper,
    maze_cells=MAZE_CELLS,
    img_size=IMG_SIZE,
    max_steps=MAX_EP_STEPS,
    device=DEVICE,
    seed=0,
)

model = PPO(
    "CnnPolicy",
    env,
    verbose=1,
    device="mps" if DEVICE.type == "mps" else DEVICE,
    learning_rate=2e-4,
    policy_kwargs={"normalize_images": False},
)

from stable_baselines3.common.logger import configure
log_dir = "logs/ppo_maze_fm_actions"
logger = configure(log_dir, ["stdout", "csv"])
model.set_logger(logger)

model.learn(total_timesteps=PPO_STEPS)


Evaluate success rate.

In [ ]:
eval_env = MazeEnvFMAction(
    stepper=stepper,
    maze_cells=MAZE_CELLS,
    img_size=IMG_SIZE,
    max_steps=MAX_EP_STEPS,
    device=DEVICE,
    seed=123,
)

success = 0
rewards = []
for _ in range(EVAL_EPISODES):
    obs, _ = eval_env.reset()
    terminated = False
    truncated = False
    ep_reward = 0.0

    while not (terminated or truncated):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = eval_env.step(int(action))
        ep_reward += reward

    rewards.append(ep_reward)
    if terminated:
        success += 1

print(f"Mean episode reward: {np.mean(rewards):.2f}")
print(f"Success rate: {success / EVAL_EPISODES:.2%}")


Plot PPO losses (same style as pipeline_tetris).

In [ ]:
import csv
import os

log_path = os.path.join(log_dir, "progress.csv")
if not os.path.exists(log_path):
    print(f"No progress.csv found at {log_path}. Did training run?")
else:
    with open(log_path, "r", newline="") as f:
        rows = list(csv.DictReader(f))

    def get_series(key: str):
        vals = []
        for r in rows:
            v = r.get(key, "")
            if v != "" and v is not None:
                vals.append(float(v))
        return vals

    policy_loss = get_series("train/policy_gradient_loss")
    value_loss = get_series("train/value_loss")
    entropy_loss = get_series("train/entropy_loss")
    total_loss = get_series("train/loss")

    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    if policy_loss:
        axes[0].plot(policy_loss, label="policy")
    if entropy_loss:
        axes[0].plot(entropy_loss, label="entropy")
    axes[0].set_title("Policy/Entropy")
    axes[0].set_xlabel("Update")
    axes[0].legend()

    if value_loss:
        axes[1].plot(value_loss, label="value")
    if total_loss:
        axes[1].plot(total_loss, label="total")
    axes[1].set_title("Value/Total")
    axes[1].set_xlabel("Update")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


Rollout GIFs (4 samples) side-by-side.

In [ ]:
import imageio.v2 as imageio
import os
import base64
from IPython.display import HTML, display


def render_env_frame(env: MazeEnvFMAction) -> np.ndarray:
    cond = env.cond_static[0].detach().cpu().numpy()
    trace = env.trace[0].detach().cpu().numpy()[0]
    walls = cond[0]
    start_ch = cond[1]
    goal_ch = cond[2]
    img = np.ones((walls.shape[0], walls.shape[1], 3), dtype=np.float32)
    img[walls > 0.5] = 0.0
    img[start_ch > 0.5] = np.array([0.2, 1.0, 0.2])
    img[goal_ch > 0.5] = np.array([0.2, 0.2, 1.0])
    img[trace > 0.5] = np.array([1.0, 0.2, 0.2])
    return (img * 255).astype(np.uint8)


def rollout_and_save_gif(env: MazeEnvFMAction, model: PPO, out_path: str, max_steps: int = 120, fps: int = 6, seed: int | None = None):
    if seed is not None:
        set_seed(seed)
    obs, _ = env.reset()
    frames = []
    for _ in range(max_steps):
        frames.append(render_env_frame(env))
        action, _ = model.predict(obs, deterministic=True)
        obs, _, terminated, truncated, _ = env.step(int(action))
        if terminated or truncated:
            frames.append(render_env_frame(env))
            break

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    imageio.mimsave(out_path, frames, fps=fps)
    return out_path


def gif_to_base64(path: str) -> str:
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("ascii")


gif_env = MazeEnvFMAction(
    stepper=stepper,
    maze_cells=MAZE_CELLS,
    img_size=IMG_SIZE,
    max_steps=MAX_EP_STEPS,
    device=DEVICE,
    seed=999,
)

num_gifs = 4
gif_paths = []
base_seed = 12345
for i in range(num_gifs):
    out_path = f"gifs/maze_ppo_fm_actions_{i+1}.gif"
    gif_paths.append(rollout_and_save_gif(gif_env, model, out_path=out_path, seed=base_seed + i))

html = "<div style='display:flex; gap:12px; flex-wrap:wrap; align-items:flex-start;'>" + "".join(
    [f"<div style='text-align:center'><img src='data:image/gif;base64,{gif_to_base64(p)}' style='width:160px; image-rendering:pixelated;'/><div style='font-size:12px'>{os.path.basename(p)}</div></div>" for p in gif_paths]
) + "</div>"

display(HTML(html))

print(gif_paths)
